In [0]:
from pyspark.sql import functions as F

# Read existing Gold tables
features_df = spark.table(
    "oil_gas_demo.gold.demand_features"
)

demand_df = spark.table(
    "oil_gas_demo.gold.daily_demand_agg"
)

print("Feature table rows:", features_df.count())
print("Daily demand rows:", demand_df.count())

display(features_df.limit(5))
display(demand_df.limit(5))

In [0]:
# Check duplicate records in daily demand

KEYS = [
    "transaction_date",
    "product_name",
    "destination_city"
]

duplicates = (
    demand_df
    .groupBy(*KEYS)
    .count()
    .filter("count > 1")
)

duplicate_count = duplicates.count()

print("Duplicate records:", duplicate_count)

display(duplicates.limit(10))

In [0]:
from pyspark.sql import functions as F

# Prepare feature data
features = (
    features_df
    .withColumn(
        "transaction_date",
        F.to_date("transaction_date")
    )
    .alias("f")
)

# Prepare next-day actual demand
actuals = (
    demand_df
    .select(
        F.to_date("transaction_date").alias("target_date"),
        "product_name",
        "destination_city",
        F.col("total_demand").alias("target_demand")
    )
    .alias("a")
)

# Match today's features with tomorrow's actual demand
training_df = (
    features
    .join(
        actuals,
        (
            (F.col("f.product_name") == F.col("a.product_name")) &
            (F.col("f.destination_city") == F.col("a.destination_city")) &
            (
                F.col("a.target_date") ==
                F.date_add(F.col("f.transaction_date"), 1)
            )
        ),
        "inner"
    )
    .select(
        "f.*",
        F.col("a.target_date"),
        F.col("a.target_demand")
    )
)

print("Training rows:", training_df.count())

display(
    training_df.select(
        "transaction_date",
        "target_date",
        "product_name",
        "destination_city",
        "total_demand",
        "target_demand"
    ).limit(20)
)

In [0]:
invalid_targets = training_df.filter(
    F.col("target_date") !=
    F.date_add(F.col("transaction_date"), 1)
)

print("Invalid target dates:", invalid_targets.count())

In [0]:
from pyspark.sql import functions as F

# Remove rows where the target demand is missing
training_clean_df = training_df.filter(
    F.col("target_demand").isNotNull()
)

# Save the 1-day-ahead training dataset
table_name = "oil_gas_demo.gold.demand_training_features_1day"

(
    training_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

print("Training dataset saved successfully!")
print("Table:", table_name)
print("Rows saved:", training_clean_df.count())

In [0]:
saved_df = spark.table(
    "oil_gas_demo.gold.demand_training_features_1day"
)

print("Saved rows:", saved_df.count())

display(
    saved_df.select(
        "transaction_date",
        "target_date",
        "product_name",
        "destination_city",
        "total_demand",
        "target_demand"
    ).limit(10)
)

In [0]:
# Load the saved 1-day training dataset
training_df = spark.table(
    "oil_gas_demo.gold.demand_training_features_1day"
)

# Display all column names
print("Available columns:")
print(training_df.columns)

# View sample records
display(training_df.limit(5))

In [0]:
from pyspark.sql import functions as F

# Load our saved 1-day forecasting dataset
training_df = spark.table(
    "oil_gas_demo.gold.demand_training_features_1day"
)

# Check the dataset's date range and size
training_df.select(
    F.min("transaction_date").alias("start_date"),
    F.max("transaction_date").alias("end_date"),
    F.count("*").alias("total_rows"),
    F.countDistinct("transaction_date").alias("unique_dates")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Get all unique target dates in chronological order
dates_df = (
    training_df
    .select("target_date")
    .distinct()
    .orderBy("target_date")
)

# 2. Count the unique target dates
total_dates = dates_df.count()

# Keep 80% of dates for training
train_date_count = int(total_dates * 0.80)

# 3. Find the last date belonging to the training period
date_window = Window.orderBy("target_date")

cutoff_date = (
    dates_df
    .withColumn(
        "date_number",
        F.row_number().over(date_window)
    )
    .filter(F.col("date_number") == train_date_count)
    .select("target_date")
    .first()["target_date"]
)

# 4. Split the dataset chronologically
train_df = training_df.filter(
    F.col("target_date") <= F.lit(cutoff_date)
)

test_df = training_df.filter(
    F.col("target_date") > F.lit(cutoff_date)
)

# 5. Display the split details
print("Total unique target dates:", total_dates)
print("Training cutoff date:", cutoff_date)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

print(
    "Training date range:",
    train_df.agg(F.min("target_date"), F.max("target_date")).first()
)

print(
    "Testing date range:",
    test_df.agg(F.min("target_date"), F.max("target_date")).first()
)

In [0]:
from pyspark.sql import functions as F

# Columns used as model inputs
feature_cols = [
    "total_demand",
    "avg_unit_price",
    "total_inventory",
    "day_of_week",
    "month",
    "demand_lag_1",
    "demand_lag_7",
    "rolling_avg_7",
    "product_name",
    "destination_city"
]

# Check missing values in the training and testing data
columns_to_check = feature_cols + ["target_demand"]

null_checks = train_df.agg(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in columns_to_check
    ]
)

print("Missing values in training data:")
null_checks.show(truncate=False)

test_null_checks = test_df.agg(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in columns_to_check
    ]
)

print("Missing values in testing data:")
test_null_checks.show(truncate=False)

In [0]:
# Features used by the model
feature_cols = [
    "total_demand",
    "avg_unit_price",
    "total_inventory",
    "day_of_week",
    "month",
    "demand_lag_1",
    "demand_lag_7",
    "rolling_avg_7",
    "product_name",
    "destination_city"
]

# Convert Spark DataFrames to pandas
train_pd = train_df.select(
    feature_cols + ["target_demand"]
).toPandas()

test_pd = test_df.select(
    feature_cols + ["target_demand"]
).toPandas()

# Separate input features (X) and target (y)
X_train = train_pd[feature_cols]
y_train = train_pd["target_demand"]

X_test = test_pd[feature_cols]
y_test = test_pd["target_demand"]

# Check the results
print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("Testing features:", X_test.shape)
print("Testing target:", y_test.shape)

display(X_train.head())

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Separate numerical and categorical columns
numeric_features = [
    "total_demand",
    "avg_unit_price",
    "total_inventory",
    "day_of_week",
    "month",
    "demand_lag_1",
    "demand_lag_7",
    "rolling_avg_7"
]

categorical_features = [
    "product_name",
    "destination_city"
]

# Create the preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numeric_features
        )
    ]
)

# Learn category mappings from training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same mappings to testing data
X_test_processed = preprocessor.transform(X_test)

# Check the processed data
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)